## USING LSTM TO PREDICT VAR

### Setting up data and sets

In [12]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Load the data 
# (Added index_col=0 so your dates stay in the index, not as a separate column)
path = r"C:\Users\nisha\Desktop\2026 PS-II\Projects\VaR\pythonProject1\data"
import_file = os.path.join(path, 'Data_with_features.csv')
sp500_data = pd.read_csv(import_file, index_col=0, parse_dates=True)

# 2. Isolate features 
features = ['Return', 'Rolling_GARCH_Vol', 'RiskMetrics_Vol', 'VIX_Daily_Shifted', 'RSI']

lstm_data = sp500_data[features].copy()
lstm_data.dropna(inplace=True)

# Calculate where the 80% split happens in the raw dataframe
split_row = int(len(lstm_data) * 0.8)
train_subset = lstm_data.iloc[:split_row]

# Initialize and FIT the scaler ONLY on the training subset
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_subset)

# Now apply that transformation scale to the entire dataset
scaled_data = scaler.transform(lstm_data)

# 4. Create Sequences (X: Past 60 days -> Y: Current day Return)
X = []
y = []
window_size = 60

for i in range(window_size, len(scaled_data)):
    X.append(scaled_data[i-window_size : i, :])
    y.append(scaled_data[i, 0])  # 0 is the column index for 'Return'
    
X, y = np.array(X), np.array(y) 

# 5. Split into Train and Test (80/20)
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"X_train shape: {X_train.shape} (Samples, Timesteps, Features)")
print(f"X_test shape:  {X_test.shape} (Samples, Timesteps, Features)")

X_train shape: (5841, 60, 5) (Samples, Timesteps, Features)
X_test shape:  (1461, 60, 5) (Samples, Timesteps, Features)


### Loss Function

In [13]:
import tensorflow as tf

# Set the VaR confidence level 
target_alpha = 0.01

# The Core Pinball/Quantile Loss Function
def quantile_loss(q, y_true, y_pred):
    error = y_true - y_pred
    return tf.reduce_mean(tf.maximum(q * error, (q - 1) * error))

# Clean Wrapper Function for Keras Compilation
def target_quantile_loss(y_true, y_pred):
    return quantile_loss(target_alpha, y_true, y_pred)

### Model

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Hyperparameters derived from the paper's optimal grid
J_UNITS = 10         # From grid
LEARNING_RATE = 0.001 # From interval [0.01, 0.001]
LAMBDA = 0.01        # Regularization parameter from their grid

model = Sequential()

# Layer 1: First Hidden Layer
model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
# Paper uses ReLU for hidden layers + Regularization
model.add(LSTM(units=J_UNITS, 
               return_sequences=True, 
               activation='relu', 
               kernel_regularizer=l2(LAMBDA)))
model.add(Dropout(0.2)) 

# Layer 2: Second Hidden Layer
model.add(LSTM(units=J_UNITS, 
               activation='relu', 
               kernel_regularizer=l2(LAMBDA)))
model.add(Dropout(0.2))

# Layer 3: Output (Linear function as per paper)
model.add(Dense(units=1, activation='linear'))

# Compile with the paper's learning rate range
optimizer = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss=target_quantile_loss)

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 60, 10)         │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 60, 10)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 10)             │           840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,491 (5.82 KB)

 Trainable params: 1,491 (5.82 KB)

 Non-trainable params: 0 (0.00 B)

### Training


In [15]:
BATCH_SIZE = 20  
EPOCHS = 50            

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,           
    batch_size=BATCH_SIZE,       
    validation_data=(X_test[20:], y_test[20:]), 
    verbose=1
)

Epoch 1/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - loss: 0.0827 - val_loss: 0.0152
Epoch 2/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - loss: 0.0063 - val_loss: 0.0024
Epoch 3/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 0.0026 - val_loss: 0.0017
Epoch 4/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 0.0025 - val_loss: 0.0021
Epoch 5/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 0.0024 - val_loss: 0.0020
Epoch 6/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - loss: 0.0024 - val_loss: 0.0019
Epoch 7/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - loss: 0.0023 - val_loss: 0.0019
Epoch 8/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - loss: 0.0024 - val_loss: 0.0019
Epoch 9/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - loss: 0.0023 - val_loss: 0.0018
Epoch 10/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - loss: 0.0023 - val_loss: 0.0017
Epoch 11/50
293/293 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - loss: 0.0024 - val_loss: 0.0018
Epoch 12/50
293/293 ━━━━━━━━━━━━━━━━━━━━

In [17]:
# 1. Generate scaled predictions
lstm_pred_scaled = model.predict(X, verbose=0)

# 2. Inverse Transform (The Dummy Matrix Trick)
# Create a 5-column matrix of zeros to satisfy the scaler's shape requirement
dummy_matrix = np.zeros((len(lstm_pred_scaled), 5))

# Inject the predictions into the first column (Index 0 = 'Return')
dummy_matrix[:, 0] = lstm_pred_scaled.flatten()

# Inverse transform and extract the un-scaled real values
lstm_pred_real = scaler.inverse_transform(dummy_matrix)[:, 0]

# 3. Assign to DataFrame cleanly
sp500_data['LSTM_VaR'] = np.nan
sp500_data.loc[sp500_data.index[window_size:], 'LSTM_VaR'] = lstm_pred_real

# 4. Verify the alignment and scale
print(sp500_data[['Return', 'LSTM_VaR']])
export_file = os.path.join(path, 'Results.csv')
sp500_data.to_csv(export_file)

              Return  LSTM_VaR
Date                          
1997-01-15 -0.216134       NaN
1997-01-16  0.331825       NaN
1997-01-17  0.830576       NaN
1997-01-20  0.068264       NaN
1997-01-21  0.772080       NaN
...              ...       ...
2026-04-15  0.794414 -3.479434
2026-04-16  0.260656 -3.479434
2026-04-17  1.196855 -3.479434
2026-04-20 -0.237720 -3.479434
2026-04-21 -0.636845 -3.479434

[7362 rows x 2 columns]
